In [31]:
import random

In [32]:
def init_pheromone(
        n_cities: int,
        init_tau: float
):
    return [[init_tau] * n_cities for _ in range(n_cities)]

In [33]:
def compute_heuristic(
        cost_matrix: list[list[int]],
):
    n = len(cost_matrix)
    eta = [[0] * n for _ in range(n)]

    for i in range(n):
        for j in range(n):
            if i != j:
                eta[i][j] = 1 / cost_matrix[i][j]

    return eta

In [34]:
def select_next_city(
        current_city: int,
        allowed_cities: list[int],
        pheromone: list[list[float]],
        heuristic: list[list[float]],
        alpha: float,
        beta: float,
):
    weights = []
    for city in allowed_cities:
        tau = pheromone[current_city][city] ** alpha
        eta = heuristic[current_city][city] ** beta
        weights.append(tau * eta)

    total_weight = sum(weights)
    probs = [weights[i] / total_weight for i in range(len(weights))]

    r = random.random()
    accumulate = 0.0
    for i in range(len(probs)):
        accumulate += probs[i]
        if r <= accumulate:
            return allowed_cities[i]

    return allowed_cities[-1]

In [35]:
def construct_tour(
        pheromone: list[list[float]],
        heuristic: list[list[float]],
        alpha: float,
        beta: float,
):
    n = len(pheromone)
    tour = [0]
    allowed_cities = [i for i in range(1, n)]
    while allowed_cities:
        current_city = tour[-1]
        next_city = select_next_city(
            current_city=current_city,
            allowed_cities=allowed_cities,
            pheromone=pheromone,
            heuristic=heuristic,
            alpha=alpha,
            beta=beta,
        )
        tour.append(next_city)
        allowed_cities.remove(next_city)

    tour.append(0)

    return tour

In [36]:
def compute_tour_length(
        cost_matrix: list[list[int]],
        tour: list[int],
):
    length = 0
    for i in range(len(tour) - 1):
        length += cost_matrix[tour[i]][tour[i + 1]]

    return length

In [37]:
def evaporate_pheromone(
        pheromone: list[list[float]],
        rho: float,
):
    n = len(pheromone)
    for i in range(n):
        for j in range(n):
            pheromone[i][j] *= (1 - rho)

def deposit_pheromone(
        pheromone: list[list[float]],
        tours: list[list[int]],
        lengths: list[int],
        pheromone_const: float
):
    for tour, length in zip(tours, lengths):
        delta = pheromone_const / length
        for i in range(len(tour) - 1):
            pheromone[tour[i]][tour[i + 1]] += delta
            pheromone[tour[i + 1]][tour[i]] += delta

In [38]:
# Main ACO
def ant_colony_optimization_tsp(
        cost_matrix: list[list[int]],
        n_ants: int,
        n_iters: int,
        init_tau: float,
        alpha: float,
        beta: float,
        rho: float,
        pheromone_const: float
):
    n = len(cost_matrix)
    pheromone = init_pheromone(n_cities=n, init_tau=init_tau)
    heuristic = compute_heuristic(cost_matrix=cost_matrix)

    best_tour = None
    best_length = float('inf')

    for i in range(n_iters):
        print(f"Iteration {i+1}/{n_iters}")
        tours = []
        lengths = []
        best_local_length = float('inf')

        for j in range(n_ants):
            tour = construct_tour(pheromone, heuristic, alpha, beta)
            length = compute_tour_length(cost_matrix, tour)

            tours.append(tour)
            lengths.append(length)

            if length < best_length:
                best_tour = tour
                best_length = length
                print(f"New best tour length: {best_length}")

            if length < best_local_length:
                best_local_length = length

        print(f"(Local in iter) New best tour length: {best_local_length}")

        evaporate_pheromone(pheromone, rho)
        deposit_pheromone(pheromone, tours, lengths, pheromone_const)

    return best_tour, best_length

In [39]:
import math

def generate_euclidean_cost_matrix(n, seed=0):
    random.seed(seed)
    points = [(random.uniform(0,100), random.uniform(0,100)) for _ in range(n)]
    cost = [[0]*n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i != j:
                dx = points[i][0] - points[j][0]
                dy = points[i][1] - points[j][1]
                cost[i][j] = round(math.hypot(dx, dy), 2)
    return cost

In [40]:
cost_matrix = generate_euclidean_cost_matrix(20)

best_tour, best_length = ant_colony_optimization_tsp(
    cost_matrix=cost_matrix,
    n_ants=100,
    n_iters=1000,
    init_tau=1.0,
    alpha=1.0,
    beta=1.0,
    rho=0.1,
    pheromone_const=1.0
)

print(">" * 20)
print(f"Best tour length: {best_length}")
print(f"Best tour: {best_tour}")
print(">" * 20)

Iteration 1/1000
New best tour length: 686.5
New best tour length: 639.4799999999999
New best tour length: 584.66
New best tour length: 532.4499999999999
(Local in iter) New best tour length: 532.4499999999999
Iteration 2/1000
New best tour length: 529.22
New best tour length: 527.12
New best tour length: 485.82
(Local in iter) New best tour length: 485.82
Iteration 3/1000
(Local in iter) New best tour length: 502.80999999999995
Iteration 4/1000
New best tour length: 482.43999999999994
New best tour length: 416.53
(Local in iter) New best tour length: 416.53
Iteration 5/1000
(Local in iter) New best tour length: 490.65000000000003
Iteration 6/1000
(Local in iter) New best tour length: 493.59000000000003
Iteration 7/1000
(Local in iter) New best tour length: 444.26
Iteration 8/1000
(Local in iter) New best tour length: 513.49
Iteration 9/1000
(Local in iter) New best tour length: 471.78
Iteration 10/1000
(Local in iter) New best tour length: 496.76
Iteration 11/1000
New best tour length